# `11_colab_l4_live_demo.ipynb` — End-to-End Live QLoRA Fine-Tuning & Test-Set Evaluation on Google Colab (L4 GPU)

This self-contained notebook is designed to run **live on a Google Colab Pro L4 GPU (`24 GB VRAM`)** (or locally on CUDA / Apple Silicon `mps`).
- **Zero live XML preprocessing needed:** The repository includes all 26,209 canonical GermEval 2017 documents in compact `data/canonical/*.jsonl.gz` archives (`5 MB` total) and automatically materializes the Task C and Task D JSONL files in ~1.5 seconds.
- **100% Pure-Python Evaluation:** Computes Strict Task C1/C2 Micro-F1, Macro-F1, Exact-Set Accuracy, Bootstrap 95% CIs, and Exact Official Java-Compatible Scores in pure Python (`src/germeval_llm/evaluation.py`).

---
## Step 1: Google Colab Bootstrap (Clone Repo, Install Package, Authenticate HF)

In [ ]:
import os, sys
from pathlib import Path

# If running on Google Colab and not yet inside the repository, clone it and install dependencies:
if 'google.colab' in sys.modules:
    REPO_URL = os.environ.get('GERMEVAL_REPO_URL', 'https://github.com/simon-clematide/gemma-qlora-finetuning.git')
    if not Path('pyproject.toml').exists() and not Path('gemma-qlora-finetuning/pyproject.toml').exists():
        print(f'Cloning {REPO_URL}...')
        os.system(f'git clone {REPO_URL}')
    if Path('gemma-qlora-finetuning/pyproject.toml').exists():
        os.chdir('gemma-qlora-finetuning')
    os.system('pip install -q -e ".[ml]"')

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
    os.chdir(ROOT)
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from germeval_llm.convert import ensure_derived_datasets_ready
from germeval_llm.inference import detect_compute_device

ensure_derived_datasets_ready()
hw = detect_compute_device()
print('Repository initialized:', ROOT.name)
print('Detected compute hardware:', hw)

### Authenticate with Hugging Face Hub (for gated `google/gemma-2-2b-it`)
In Google Colab, store your Hugging Face token in **Colab Secrets** (key icon on the left sidebar) under the name `HF_TOKEN`. The cell below automatically logs in using `HF_TOKEN` from Colab Secrets or environment variables.

In [ ]:
import os
hf_token = os.environ.get('HF_TOKEN')
if not hf_token and 'google.colab' in sys.modules:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
    except Exception:
        pass
if hf_token:
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print('Authenticated with Hugging Face Hub via HF_TOKEN.')
else:
    print('Using existing local Hugging Face credentials (or run `huggingface_hub.login()` if needed).')

---
## Step 2: Inspect the Dataset & Completion-Only Loss Masking
Before loading the model, let's inspect a real training example and verify how our prompt and completion are formatted for `google/gemma-2-2b-it`. Remember:
- **Task C Output Contract:** `{"labels": ["Category:polarity", ...]}` (or `{"labels": []}` when `relevance=false`).
- **Completion-Only Loss (`labels = -100`):** During QLoRA training, all prompt tokens up to `<start_of_turn>model\n` are masked with `-100` so gradients only update on the JSON completion.

In [ ]:
import json
import pandas as pd
from IPython.display import display
from germeval_llm.xml_parser import load_canonical_jsonl
from germeval_llm.schemas import TaskCExample
from germeval_llm.prompts import format_gemma_chat_string, compute_completion_only_labels

train_docs = load_canonical_jsonl('data/canonical/train.jsonl')
dev_docs = load_canonical_jsonl('data/canonical/dev.jsonl')
test_syn_docs = load_canonical_jsonl('data/canonical/test_syn.jsonl')
test_dia_docs = load_canonical_jsonl('data/canonical/test_dia.jsonl')
print(f'Loaded splits -> train: {len(train_docs)}, dev: {len(dev_docs)}, test_syn: {len(test_syn_docs)}, test_dia: {len(test_dia_docs)}')

with Path('data/derived/task_c/train.jsonl').open() as f:
    sample_ex = TaskCExample.model_validate_json(f.readline())

print('\nSample Document Text:', sample_ex.text)
print('Target JSON Completion:', sample_ex.completion[0].content)

---
## Step 3: Configure Experiment Scope for Live Demo
On an **NVIDIA L4 GPU**, you can either run a **fast 5-minute live demo** (`DEMO_MODE = 'fast_l4'`, training on `N=1,000` stratified examples for `150` steps and evaluating on `200` `test_syn` documents) or the **full production run** (`DEMO_MODE = 'full_l4'`, training on all `19,432` examples and evaluating on all `2,566` `test_syn` documents).

In [ ]:
import torch

# Set RUN_LIVE_GPU = True on Google Colab ( automatically enabled when CUDA GPU is present! )
RUN_LIVE_GPU = torch.cuda.is_available()
DEMO_MODE = 'fast_l4'  # Choose 'fast_l4' (~5 mins on L4) or 'full_l4' (~45 mins on L4)

if DEMO_MODE == 'fast_l4':
    TRAIN_SUBSET_SIZE = 1000
    MAX_TRAIN_STEPS = 120
    EVAL_MAX_DOCS = 150
else:
    TRAIN_SUBSET_SIZE = None  # All 19,432 train docs
    MAX_TRAIN_STEPS = None    # Full 3 epochs (3,644 steps)
    EVAL_MAX_DOCS = None      # All 2,566 test_syn docs

print(f'RUN_LIVE_GPU={RUN_LIVE_GPU} | DEMO_MODE={DEMO_MODE} | TRAIN_SUBSET_SIZE={TRAIN_SUBSET_SIZE} | MAX_TRAIN_STEPS={MAX_TRAIN_STEPS} | EVAL_MAX_DOCS={EVAL_MAX_DOCS}')

---
## Step 4: Baseline Inference on `test_syn` (`test_TIMESTAMP1`) Before Fine-Tuning
First, let's evaluate the frozen `google/gemma-2-2b-it` model with our guideline prompt (`P1`) on the synchronic test set (`test_syn`) and score it with our **pure-Python evaluator** (`evaluate_task_c_predictions`).

In [ ]:
from germeval_llm.inference import (
    load_gemma_model_and_tokenizer,
    build_hf_generator_fn,
    run_task_c_inference_with_generator,
    save_predictions_jsonl,
    load_predictions_jsonl,
)
from germeval_llm.evaluation import evaluate_task_c_predictions

eval_docs = test_syn_docs[:EVAL_MAX_DOCS] if EVAL_MAX_DOCS else test_syn_docs

if RUN_LIVE_GPU:
    base_model, tokenizer = load_gemma_model_and_tokenizer(
        model_id='google/gemma-2-2b-it',
        load_in_4bit=True,
    )
    gen_fn = build_hf_generator_fn(base_model, tokenizer, max_new_tokens=256)
    baseline_preds = run_task_c_inference_with_generator(
        eval_docs,
        run_id='colab_baseline_p1',
        split='test_syn',
        generate_fn=gen_fn,
        prompt_key='p1_guidelines',
    )
    save_predictions_jsonl(baseline_preds, 'results/predictions/colab_baseline_p1_test_syn.jsonl')
    baseline_metrics = evaluate_task_c_predictions(eval_docs, baseline_preds)
    del base_model
    torch.cuda.empty_cache()
else:
    print('CUDA GPU not active in offline build; loading verified smoke baseline metrics:')
    baseline_metrics = json.loads(Path('results/metrics/smoke_p1_dev_dev.json').read_text())

print('Baseline (Zero-Shot P1) Summary:')
print(json.dumps({
    'num_documents': baseline_metrics['num_documents'],
    'schema_validity_rate': baseline_metrics['schema_validity_rate'],
    'valid_exact_rate': baseline_metrics['valid_exact_rate'],
    'parse_status_counts': baseline_metrics['parse_status_counts'],
    'c1_strict_micro_f1': round(baseline_metrics['task_c1_category_strict']['micro_f1'], 4),
    'c2_strict_micro_f1': round(baseline_metrics['task_c2_category_sentiment_strict']['micro_f1'], 4),
    'c2_java_compat_micro_f1': round(baseline_metrics['task_c2_category_sentiment_java_compat']['micro_f1'], 4),
}, indent=2))

---
## Step 5: Live 4-bit NF4 QLoRA Fine-Tuning (`trl.SFTTrainer`)
Now we train a 4-bit NF4 QLoRA adapter ($r=16, \alpha=32$, targeting all 7 linear projections `q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj`) with `completion_only_loss=True`:

In [ ]:
from germeval_llm.training import run_sft_adapter_training

ADAPTER_DIR = 'models/adapters/colab_qlora_task_c'

if RUN_LIVE_GPU:
    train_summary = run_sft_adapter_training(
        config_path='configs/qlora.yaml',
        train_jsonl_path='data/derived/task_c/train.jsonl',
        dev_jsonl_path='data/derived/task_c/dev.jsonl',
        output_adapter_dir=ADAPTER_DIR,
        subset_size=TRAIN_SUBSET_SIZE,
        max_steps_override=MAX_TRAIN_STEPS,
        sampling_strategy='stratified',
        seed_override=42,
    )
    torch.cuda.empty_cache()
else:
    train_summary = json.loads(Path('results/runs/smoke_task_c_lora_train_summary.json').read_text())

print('Adapter Training Summary:')
print(json.dumps(train_summary, indent=2))

---
## Step 6: Evaluate the Fine-Tuned QLoRA Adapter on `test_syn` & Compare Side-by-Side
Now we reload the saved adapter (`models/adapters/colab_qlora_task_c`) onto the 4-bit base model, run greedy inference on the exact same `test_syn` split, and compare Before vs. After metrics in pure Python:

In [ ]:
if RUN_LIVE_GPU:
    ft_model, tokenizer = load_gemma_model_and_tokenizer(
        model_id='google/gemma-2-2b-it',
        adapter_path=ADAPTER_DIR,
        load_in_4bit=True,
    )
    ft_gen_fn = build_hf_generator_fn(ft_model, tokenizer, max_new_tokens=256)
    qlora_preds = run_task_c_inference_with_generator(
        eval_docs,
        run_id='colab_qlora_r16',
        split='test_syn',
        generate_fn=ft_gen_fn,
        prompt_key='p1_guidelines',
    )
    save_predictions_jsonl(qlora_preds, 'results/predictions/colab_qlora_r16_test_syn.jsonl')
    qlora_metrics = evaluate_task_c_predictions(eval_docs, qlora_preds)
    del ft_model
    torch.cuda.empty_cache()
else:
    qlora_metrics = json.loads(Path('results/metrics/smoke_lora_dev_dev.json').read_text())

comparison_df = pd.DataFrame([
    {
        'Model': '1. Frozen Gemma-2-2B-IT (Zero-Shot P1)',
        'Valid Exact JSON Rate': baseline_metrics['valid_exact_rate'],
        'Schema Validity Rate': baseline_metrics['schema_validity_rate'],
        'Task C1 Strict Micro-F1': round(baseline_metrics['task_c1_category_strict']['micro_f1'], 4),
        'Task C2 Strict Micro-F1': round(baseline_metrics['task_c2_category_sentiment_strict']['micro_f1'], 4),
        'Task C2 Official Java-Compat F1': round(baseline_metrics['task_c2_category_sentiment_java_compat']['micro_f1'], 4),
        'Mean Latency (ms/doc)': round(baseline_metrics['efficiency']['mean_latency_ms_per_doc'], 1),
    },
    {
        'Model': '2. Fine-Tuned Gemma-2-2B-IT (QLoRA r=16)',
        'Valid Exact JSON Rate': qlora_metrics['valid_exact_rate'],
        'Schema Validity Rate': qlora_metrics['schema_validity_rate'],
        'Task C1 Strict Micro-F1': round(qlora_metrics['task_c1_category_strict']['micro_f1'], 4),
        'Task C2 Strict Micro-F1': round(qlora_metrics['task_c2_category_sentiment_strict']['micro_f1'], 4),
        'Task C2 Official Java-Compat F1': round(qlora_metrics['task_c2_category_sentiment_java_compat']['micro_f1'], 4),
        'Mean Latency (ms/doc)': round(qlora_metrics['efficiency']['mean_latency_ms_per_doc'], 1),
    },
])
display(comparison_df)

---
## Step 7: Qualitative Inspection of Predictions Before vs. After Fine-Tuning
Let's inspect the raw strings emitted by the base model vs. the fine-tuned adapter on individual documents:

In [ ]:
if RUN_LIVE_GPU:
    b_preds = load_predictions_jsonl('results/predictions/colab_baseline_p1_test_syn.jsonl')
    q_preds = load_predictions_jsonl('results/predictions/colab_qlora_r16_test_syn.jsonl')
    docs_slice = eval_docs[:5]
else:
    b_preds = load_predictions_jsonl('results/predictions/smoke_p1_dev_dev.jsonl')
    q_preds = load_predictions_jsonl('results/predictions/smoke_lora_dev_dev.jsonl')
    docs_slice = dev_docs[:5]

from germeval_llm.convert import extract_task_c_label_sets
for doc, bp, qp in zip(docs_slice, b_preds[:5], q_preds[:5]):
    gold_c2, _ = extract_task_c_label_sets(doc, apply_typo_patches=True)
    print('=' * 90)
    print('Text:', doc.text[:160])
    print('Gold C2 Labels :', gold_c2)
    print(f'Zero-Shot Raw  ({bp.parse_status}): {bp.raw_output!r}')
    print(f'Fine-Tuned Raw ({qp.parse_status}): {qp.raw_output!r}')